# California Housing Price Prediction — Linear vs Non-Linear Models

## Objective
Build a high-accuracy regression system to predict median house value using rich tabular housing features.
Compare Linear Regression, Lasso Regression, and Random Forest Regressor with a strong focus on:
- Model accuracy and robustness
- Data quality and feature engineering
- Interpretability vs pure performance

## Dataset
Using the California Housing dataset from scikit-learn with:
- 20,640 rows
- 8 features + 1 target variable
- Minimal missing values

## 1. Exploratory Data Analysis (EDA) & Data Quality

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load the California Housing dataset
california_housing = fetch_california_housing()

# Convert to DataFrame
data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
data['median_house_value'] = california_housing.target

# Display basic information
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())
print("\nFirst 5 rows:")
data.head()

In [ ]:
# Check data types and basic info
data.info()

In [ ]:
# Check for missing values
print("Missing values:")
data.isnull().sum()

In [ ]:
# Summary statistics
data.describe()

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 6))
plt.hist(data['median_house_value'], bins=50, edgecolor='black')
plt.title('Distribution of Median House Value')
plt.xlabel('Median House Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(12, 10))
correlation_matrix = data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Pairplot for key features
key_features = ['MedInc', 'AveRooms', 'HouseAge', 'median_house_value']
sns.pairplot(data[key_features])
plt.show()

In [ ]:
# Boxplots to detect outliers
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for idx, column in enumerate(data.columns):
    sns.boxplot(y=data[column], ax=axes[idx])
    axes[idx].set_title(f'{column} Boxplot')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
# Create new interaction features
data['rooms_per_household'] = data['AveRooms'] / data['AveOccup']
data['bedrooms_ratio'] = data['AveBedrms'] / data['AveRooms']
data['population_per_household'] = data['Population'] / data['AveOccup']
data['rooms_per_person'] = data['AveRooms'] / data['Population']

# Handle potential division by zero
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.fillna(data.median(), inplace=True)

print("New features created:")
print("- rooms_per_household: Total rooms per household")
print("- bedrooms_ratio: Proportion of bedrooms to total rooms")
print("- population_per_household: Average people per household")
print("- rooms_per_person: Rooms per person")

# Check the updated dataset
data.head()

In [ ]:
# Check correlation of new features with target
new_corr = data[['rooms_per_household', 'bedrooms_ratio', 'population_per_household', 
                 'rooms_per_person', 'median_house_value']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(new_corr, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation of New Features with Target')
plt.show()

## 3. Train/Validation/Test Split

In [ ]:
# Define features and target
X = data.drop('median_house_value', axis=1)
y = data['median_house_value']

# Split the data: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 ≈ 0.15/0.85

print(f"Training set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")

## 4. Baseline: Linear Regression (with Scaling)

In [ ]:
# Create Linear Regression pipeline with StandardScaler
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Train the model
lr_pipeline.fit(X_train, y_train)

# Make predictions
lr_train_pred = lr_pipeline.predict(X_train)
lr_val_pred = lr_pipeline.predict(X_val)
lr_test_pred = lr_pipeline.predict(X_test)

print("Linear Regression model trained successfully!")

In [ ]:
# Evaluate Linear Regression
def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    }

# Calculate metrics for Linear Regression
lr_train_metrics = calculate_metrics(y_train, lr_train_pred, 'Linear Regression (Train)')
lr_val_metrics = calculate_metrics(y_val, lr_val_pred, 'Linear Regression (Validation)')
lr_test_metrics = calculate_metrics(y_test, lr_test_pred, 'Linear Regression (Test)')

# Create comparison DataFrame
lr_metrics_df = pd.DataFrame([lr_train_metrics, lr_val_metrics, lr_test_metrics])
print("Linear Regression Performance:")
lr_metrics_df

In [ ]:
# Residual analysis for Linear Regression
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs predicted values
lr_residuals = y_test - lr_test_pred
axes[0].scatter(lr_test_pred, lr_residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Linear Regression: Residuals vs Predicted Values')

# Distribution of residuals
axes[1].hist(lr_residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Linear Regression: Distribution of Residuals')

plt.tight_layout()
plt.show()

## 5. Lasso Regression (Feature Selection + Regularization)

In [ ]:
# Create Lasso pipeline with StandardScaler
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Lasso(random_state=42))
])

# Perform hyperparameter tuning for alpha
print("Performing hyperparameter tuning for Lasso...")
lasso_param_grid = {
    'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

lasso_grid_search = GridSearchCV(
    lasso_pipeline,
    lasso_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

# Fit the grid search
lasso_grid_search.fit(X_train, y_train)

# Get the best model
best_lasso_model = lasso_grid_search.best_estimator_

# Make predictions
lasso_train_pred = best_lasso_model.predict(X_train)
lasso_val_pred = best_lasso_model.predict(X_val)
lasso_test_pred = best_lasso_model.predict(X_test)

print("\nBest Lasso parameters:")
print(lasso_grid_search.best_params_)
print("\nLasso model trained successfully!")

In [ ]:
# Evaluate Lasso Regression
# Calculate metrics for Lasso Regression
lasso_train_metrics = calculate_metrics(y_train, lasso_train_pred, 'Lasso Regression (Train)')
lasso_val_metrics = calculate_metrics(y_val, lasso_val_pred, 'Lasso Regression (Validation)')
lasso_test_metrics = calculate_metrics(y_test, lasso_test_pred, 'Lasso Regression (Test)')

# Create comparison DataFrame
lasso_metrics_df = pd.DataFrame([lasso_train_metrics, lasso_val_metrics, lasso_test_metrics])
print("Lasso Regression Performance:")
lasso_metrics_df

In [ ]:
# Feature importance for Lasso (coefficients)
lasso_coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': best_lasso_model.named_steps['regressor'].coef_
})
lasso_coefficients['Abs_Coefficient'] = np.abs(lasso_coefficients['Coefficient'])
lasso_coefficients = lasso_coefficients.sort_values('Abs_Coefficient', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=lasso_coefficients, x='Coefficient', y='Feature')
plt.title('Lasso Regression Coefficients')
plt.xlabel('Coefficient Value')
plt.show()

print("Lasso Regression Coefficients:")
lasso_coefficients[['Feature', 'Coefficient']]

In [ ]:
# Residual analysis for Lasso Regression
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs predicted values
lasso_residuals = y_test - lasso_test_pred
axes[0].scatter(lasso_test_pred, lasso_residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Lasso Regression: Residuals vs Predicted Values')

# Distribution of residuals
axes[1].hist(lasso_residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Lasso Regression: Distribution of Residuals')

plt.tight_layout()
plt.show()

## 6. Random Forest Regressor (Non-Linear Model)

In [ ]:
# Create Random Forest pipeline
rf_pipeline = Pipeline([
    ('regressor', RandomForestRegressor(random_state=42))
])

# Perform hyperparameter tuning
print("Performing hyperparameter tuning for Random Forest...")
rf_param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__max_features': ['sqrt', 'auto']
}

rf_grid_search = GridSearchCV(
    rf_pipeline,
    rf_param_grid,
    cv=3,  # Reduced for faster execution
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Fit the grid search (this may take a few minutes)
rf_grid_search.fit(X_train, y_train)

# Get the best model
best_rf_model = rf_grid_search.best_estimator_

# Make predictions
rf_train_pred = best_rf_model.predict(X_train)
rf_val_pred = best_rf_model.predict(X_val)
rf_test_pred = best_rf_model.predict(X_test)

print("\nBest Random Forest parameters:")
print(rf_grid_search.best_params_)
print("\nRandom Forest model trained successfully!")

In [ ]:
# Evaluate Random Forest
# Calculate metrics for Random Forest
rf_train_metrics = calculate_metrics(y_train, rf_train_pred, 'Random Forest (Train)')
rf_val_metrics = calculate_metrics(y_val, rf_val_pred, 'Random Forest (Validation)')
rf_test_metrics = calculate_metrics(y_test, rf_test_pred, 'Random Forest (Test)')

# Create comparison DataFrame
rf_metrics_df = pd.DataFrame([rf_train_metrics, rf_val_metrics, rf_test_metrics])
print("Random Forest Performance:")
rf_metrics_df

In [ ]:
# Feature importance for Random Forest
rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_rf_model.named_steps['regressor'].feature_importances_
})
rf_importance = rf_importance.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=rf_importance, x='Importance', y='Feature')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.show()

print("Random Forest Feature Importance:")
rf_importance

In [ ]:
# Residual analysis for Random Forest
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs predicted values
rf_residuals = y_test - rf_test_pred
axes[0].scatter(rf_test_pred, rf_residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Random Forest: Residuals vs Predicted Values')

# Distribution of residuals
axes[1].hist(rf_residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Random Forest: Distribution of Residuals')

plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
# Combine all test metrics for comparison
all_test_metrics = pd.DataFrame([lr_test_metrics, lasso_test_metrics, rf_test_metrics])
print("Model Performance Comparison (Test Set):")
all_test_metrics

In [ ]:
# Visualization: Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE comparison
models = ['Linear\nRegression', 'Lasso\nRegression', 'Random\nForest']
rmse_values = [lr_test_metrics['RMSE'], lasso_test_metrics['RMSE'], rf_test_metrics['RMSE']]
axes[0].bar(models, rmse_values, color=['blue', 'green', 'red'])
axes[0].set_title('RMSE Comparison')
axes[0].set_ylabel('RMSE')

# MAE comparison
mae_values = [lr_test_metrics['MAE'], lasso_test_metrics['MAE'], rf_test_metrics['MAE']]
axes[1].bar(models, mae_values, color=['blue', 'green', 'red'])
axes[1].set_title('MAE Comparison')
axes[1].set_ylabel('MAE')

# R² comparison
r2_values = [lr_test_metrics['R²'], lasso_test_metrics['R²'], rf_test_metrics['R²']]
axes[2].bar(models, r2_values, color=['blue', 'green', 'red'])
axes[2].set_title('R² Comparison')
axes[2].set_ylabel('R²')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Predicted vs Actual values for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Linear Regression
axes[0].scatter(y_test, lr_test_pred, alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Linear Regression\nR² = {lr_test_metrics["R²"]:.4f}')

# Lasso Regression
axes[1].scatter(y_test, lasso_test_pred, alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title(f'Lasso Regression\nR² = {lasso_test_metrics["R²"]:.4f}')

# Random Forest
axes[2].scatter(y_test, rf_test_pred, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[2].set_xlabel('Actual Values')
axes[2].set_ylabel('Predicted Values')
axes[2].set_title(f'Random Forest\nR² = {rf_test_metrics["R²"]:.4f}')

plt.tight_layout()
plt.show()

## 8. Cross-Validation for Robust Performance Estimation

In [ ]:
# Perform cross-validation for all models
from sklearn.model_selection import cross_validate

# Linear Regression with scaling
lr_cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Lasso with scaling
lasso_cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Lasso(alpha=lasso_grid_search.best_params_['regressor__alpha']))
])

# Random Forest
rf_cv_pipeline = Pipeline([
    ('regressor', RandomForestRegressor(
        n_estimators=rf_grid_search.best_params_['regressor__n_estimators'],
        max_depth=rf_grid_search.best_params_['regressor__max_depth'],
        min_samples_split=rf_grid_search.best_params_['regressor__min_samples_split'],
        min_samples_leaf=rf_grid_search.best_params_['regressor__min_samples_leaf'],
        max_features=rf_grid_search.best_params_['regressor__max_features'],
        random_state=42
    ))
])

# Perform cross-validation
scoring = ['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2']

lr_cv_results = cross_validate(lr_cv_pipeline, X_train, y_train, cv=5, scoring=scoring)
lasso_cv_results = cross_validate(lasso_cv_pipeline, X_train, y_train, cv=5, scoring=scoring)
rf_cv_results = cross_validate(rf_cv_pipeline, X_train, y_train, cv=5, scoring=scoring)

# Calculate mean and std for each metric
cv_results = pd.DataFrame({
    'Model': ['Linear Regression', 'Lasso Regression', 'Random Forest'],
    'RMSE_mean': [
        -lr_cv_results['test_neg_root_mean_squared_error'].mean(),
        -lasso_cv_results['test_neg_root_mean_squared_error'].mean(),
        -rf_cv_results['test_neg_root_mean_squared_error'].mean()
    ],
    'RMSE_std': [
        lr_cv_results['test_neg_root_mean_squared_error'].std(),
        lasso_cv_results['test_neg_root_mean_squared_error'].std(),
        rf_cv_results['test_neg_root_mean_squared_error'].std()
    ],
    'MAE_mean': [
        -lr_cv_results['test_neg_mean_absolute_error'].mean(),
        -lasso_cv_results['test_neg_mean_absolute_error'].mean(),
        -rf_cv_results['test_neg_mean_absolute_error'].mean()
    ],
    'MAE_std': [
        lr_cv_results['test_neg_mean_absolute_error'].std(),
        lasso_cv_results['test_neg_mean_absolute_error'].std(),
        rf_cv_results['test_neg_mean_absolute_error'].std()
    ],
    'R2_mean': [
        lr_cv_results['test_r2'].mean(),
        lasso_cv_results['test_r2'].mean(),
        rf_cv_results['test_r2'].mean()
    ],
    'R2_std': [
        lr_cv_results['test_r2'].std(),
        lasso_cv_results['test_r2'].std(),
        rf_cv_results['test_r2'].std()
    ]
})

print("Cross-Validation Results (5-fold):")
cv_results

## 9. Summary and Insights

### Model Performance Comparison

Based on our evaluation, here's how the models performed:

1. **Random Forest** achieved the best performance with the highest R² and lowest RMSE/MAE
2. **Lasso Regression** performed similarly to Linear Regression but with automatic feature selection
3. **Linear Regression** provided a good baseline with interpretable coefficients

### Key Drivers of House Prices

From our feature importance analysis:

1. **Median Income (MedInc)** - The strongest predictor across all models
2. **Geographic Features (Latitude, Longitude)** - Important for location-based pricing
3. **Rooms per Household** - Our engineered feature showing housing size impact
4. **Housing Age** - Older homes tend to have different pricing characteristics

### Trade-offs: Interpretability vs Accuracy

1. **Linear & Lasso Regression**:
   - High interpretability with clear coefficient meanings
   - Lasso provides automatic feature selection
   - Good baseline performance

2. **Random Forest**:
   - Superior accuracy and robustness
   - Captures complex non-linear relationships
   - Less interpretable but more powerful

### Limitations and Future Improvements

1. **Data Limitations**:
   - Dataset is from 1990, may not reflect current housing markets
   - Limited features (no amenities, school districts, etc.)

2. **Model Improvements**:
   - Try more advanced models like XGBoost or Neural Networks
   - Include more feature engineering (polynomial features, interactions)
   - Handle outliers more systematically

3. **Evaluation Enhancements**:
   - Use time-based splits for temporal data
   - Include more diverse evaluation metrics
   - Perform more extensive hyperparameter tuning

## 10. Model Export

In [ ]:
# Export the best model (Random Forest) for deployment
import joblib

# Save the model
joblib.dump(best_rf_model, 'california_housing_rf_model.pkl')
print("Random Forest model saved as 'california_housing_rf_model.pkl'")

# Save the Lasso model as well
joblib.dump(best_lasso_model, 'california_housing_lasso_model.pkl')
print("Lasso model saved as 'california_housing_lasso_model.pkl'")

# Save the Linear Regression model
joblib.dump(lr_pipeline, 'california_housing_lr_model.pkl')
print("Linear Regression model saved as 'california_housing_lr_model.pkl'")

# Save the feature names for future reference
feature_names = list(X.columns)
joblib.dump(feature_names, 'feature_names.pkl')
print("Feature names saved as 'feature_names.pkl'")